# Lab 00: Gemini API Quickstart (Gemini 3.1 Edition)

Welcome to the first lab of the Google AI / Gemini series. In this notebook, we will cover the basics of setting up your environment, authenticating with the API, and making your first generation calls using the latest **Gemini 3.1** models via the `google-genai` SDK.

## Objectives
1. Install and configure the `google-genai` library.
2. Explore the latest models (Gemini 3.1 Pro & Flash-Lite).
3. Perform basic text generation and streaming.
4. Control output with **Stop Sequences**.
5. Implement **Asynchronous calls** for high-performance applications.

In [ ]:
import asyncio
import os

from dotenv import load_dotenv
from google import genai
from google.genai import types

load_dotenv()
client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))

print("Libraries imported and Client initialized.")

## 1. Exploring Available Models

As of March 2026, Google provides the **Gemini 3.1** series:
- **Gemini 3.1 Pro**: complex reasoning and coding.
- **Gemini 3.1 Flash-Lite**: Ultra-fast and cost-effective.

Let's list the models available to your API key.

In [ ]:
print("Available models supporting content generation:")
for model in client.models.list():
    if "generateContent" in model.supported_actions:
        print(f"- {model.name}")

## 2. Basic Text Generation

We use `client.models.generate_content` for standard calls.

In [ ]:
response = client.models.generate_content(
    model="gemini-3.1-flash-lite-preview",
    contents="Explain quantum computing in one sentence."
)

print(f"Response: {response.text}")

## 3. Streaming Responses

Streaming allows you to display the response as it is being generated, improving the perceived latency.

In [ ]:
prompt = "Write a short poem about the future of AI."

for chunk in client.models.generate_content_stream(
    model="gemini-3.1-flash-lite-preview",
    contents=prompt,
):
    print(chunk.text, end="", flush=True)

## 4. Controlling Output with Stop Sequences

Stop sequences tell the model to stop generating as soon as a specific string is encountered. This is very useful for formatting or preventing the model from generating unnecessary boilerplate.

In [ ]:
# Example: Stop the model before it starts a new section marked by '---'
response = client.models.generate_content(
    model="gemini-3.1-flash-lite-preview",
    config=types.GenerateContentConfig(
        stop_sequences=["---"],
        max_output_tokens=200
    ),
    contents="Generate a list of 3 fruit names. Separate them with dots." \
    "End the list with '---' and then write a long description of the first fruit."
)

print("Output with Stop Sequence:")
print(response.text)
print(
    "\nNote: The description was not generated because the model hit the stop sequence."
)

## 5. Asynchronous Content Generation

In production, you often need to handle multiple requests simultaneously. The `google-genai` SDK provides an asynchronous client (`client.aio`) for this purpose.

In [ ]:
async def generate_multiple_topics():
    topics = ["Python", "Rust", "Go"]
    tasks = []

    print(f"Launching {len(topics)} asynchronous requests...")

    for topic in topics:
        # Use client.aio for asynchronous calls
        tasks.append(client.aio.models.generate_content(
            model="gemini-3.1-flash-lite-preview",
            contents=f"Define the programming language {topic} in 10 words."
        ))

    # Wait for all tasks to complete
    responses = await asyncio.gather(*tasks)

    for topic, resp in zip(topics, responses, strict=False):
        print(f"\n[{topic}]: {resp.text}")

# In a Jupyter notebook, we can await directly if the loop is already running
await generate_multiple_topics()

## Summary

In this lab, you covered the foundations of Gemini 3.1:
1. Client initialization and model listing.
2. Synchronous and streaming generation.
3. Fine-tuning the end of generation with **Stop Sequences**.
4. Scaling with **Asynchronous calls** using `client.aio`.